# Python Method Overriding — Advanced Tutorial Laboratory, Volume II

### 16 NEW worked problems · step-by-step Jupyter lessons · explanations before solutions

**Main topic:** Python inheritance, overriding, dynamic dispatch, `__str__`, `__repr__`, and responsible use of `super()`.

This is a **different set of scenarios** from Volume I. The structure follows the supplied introductory notebook: a short narrative, a small class, a separate call cell, a question about the result, an explanation, and then a harder twist. We retain this rhythm all the way into advanced cases.

**How to study:** Run cells in order from a fresh kernel. At each **Predict** prompt, write down your guess before executing the following cell. Read each explanation *after* observing output. In each problem, the final assertions are executable answer checks; do not just read the completed solution. Problems have distinct `P01...P16` class names to avoid accidental collisions.

**Prerequisites:** inheritance, instance methods, `self`, basic exceptions, and `assert`. Only the Python standard library is needed (Python 3.9+ recommended). Expected error demonstrations catch exceptions so the notebook runs from top to bottom without stopping.

## A small lookup map to carry through the notebook

When `obj.method()` is evaluated, ordinary attribute access finds a method (or another attribute) using the object's class and its inheritance chain, then binds callable methods to `obj`. Inside a method, **`self` still refers to the original runtime object** even if the method's code was written in a parent class. Thus an inherited method that calls `self.hook()` can reach a subclass override.

A useful contrast: `super().hook()` searches *after the class where that `super()` expression was written* in the current object's method resolution order (MRO). It does not simply mean “call my immediate parent by name.” Python also has special lookup behavior for implicit operators and `str(obj)`, explored below. These are models to test with small snippets, not rules to memorize without observing behavior.

In [1]:
class LookupWarmup:
    def info(self):
        return f"base method, runtime class = {type(self).__name__}"

class LookupChild(LookupWarmup):
    pass

print(LookupChild().info())
print([cls.__name__ for cls in LookupChild.__mro__])

base method, runtime class = LookupChild
['LookupChild', 'LookupWarmup', 'object']


---

# Part I — Representation methods under pressure

We start at the precise point reached by the original lecture: overriding `__str__`, allowing `object.__str__` to fall back to `__repr__`, and remembering that lookup continues from the actual class.

## Problem 01 — The `__str__` contract is stricter than it looks

**What you will learn:** special-method contracts; debugging a failing override

**The problem.** A subclass is intended to display a measurement, but returns a number from `__str__`. Predict exactly when the error occurs and correct the design.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Read the broken implementation

Defining the class is legal: Python does not inspect the return type of this method at definition time. The error happens only when a caller needs a string.

In [2]:
class P01Measurement:
    def __init__(self, amount):
        self.amount = amount

    def __repr__(self):
        return f"P01Measurement({self.amount!r})"

class P01BrokenMeasurement(P01Measurement):
    def __str__(self):
        return self.amount  # incorrect: __str__ must return str

bad = P01BrokenMeasurement(12.5)
print("repr works:", repr(bad))

repr works: P01Measurement(12.5)


### Predict before running

Will `str(bad)` return `12.5`, return `"12.5"`, use the parent representation, or raise an error? We catch the expected exception to preserve a runnable notebook.

In [3]:
try:
    print(str(bad))
except TypeError as exc:
    print(type(exc).__name__, str(exc))

TypeError __str__ returned non-string (type float)


### Step 2 — Explain the failure

The subclass explicitly overrides `__str__`, so Python uses that method. It does **not** ignore a broken return value and try the inherited `__repr__`. A `__str__` implementation must return an actual `str`.

### Worked solution — reasoning before code

Keep `repr` useful for debugging and create a human-friendly string explicitly. `:g` is an optional compact numeric format; the essential fix is returning a string. We also deliberately test negative and zero values, rather than a single happy path.

In [4]:
class P01MeasurementFixed(P01Measurement):
    def __str__(self):
        return f"{self.amount:g} units"

for value in (12.5, 0, -3.25):
    item = P01MeasurementFixed(value)
    print("str:", str(item), "| repr:", repr(item))

str: 12.5 units | repr: P01Measurement(12.5)
str: 0 units | repr: P01Measurement(0)
str: -3.25 units | repr: P01Measurement(-3.25)


### Verify Problem 01

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [5]:
assert str(P01MeasurementFixed(12.5)) == "12.5 units"
assert str(P01MeasurementFixed(0)) == "0 units"
assert repr(P01MeasurementFixed(-3.25)) == "P01Measurement(-3.25)"
assert isinstance(str(P01MeasurementFixed(1)), str)
print("Problem 01: all checks passed")

Problem 01: all checks passed


### Check your understanding

Why did defining `__repr__` on the parent not save the broken `__str__`? **Answer:** an explicit `__str__` was found and invoked; its contract violation raised `TypeError`. The `object.__str__` fallback is relevant only when it is the selected implementation.

## Problem 02 — Why putting an object in a list changes its appearance

**What you will learn:** `str` versus `repr` in nested containers

**The problem.** An invoice line has a friendly display and an unambiguous debugging representation. Predict the output of `print(line)`, `print([line])`, and an f-string with `!r`.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Create two representations

The plain `str` uses friendly wording. The `repr` uses `!r` on the item name so quotes and escape characters remain unambiguous.

In [6]:
class P02InvoiceLine:
    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity

    def __str__(self):
        return f"{self.quantity} × {self.name}"

    def __repr__(self):
        return f"P02InvoiceLine({self.name!r}, {self.quantity!r})"

line = P02InvoiceLine('A "special" pen', 2)

### Predict before running

A container does not print every element with `str(element)`. Write down whether each of the following four expressions selects `__str__` or `__repr__`.

In [7]:
print("direct:", line)
print("inside list:", [line])
print("f-string !s:", f"{line!s}")
print("f-string !r:", f"{line!r}")

direct: 2 × A "special" pen
inside list: [P02InvoiceLine('A "special" pen', 2)]
f-string !s: 2 × A "special" pen
f-string !r: P02InvoiceLine('A "special" pen', 2)


### Step 2 — Explain the surprising part

`print(line)` uses the object’s string conversion, while the list’s own representation incorporates `repr(line)`. Explicit `!s` and `!r` conversion flags let you choose the route. Consequently, a good `__repr__` matters even for classes used mostly in containers.

### Step 3 — Extend the exercise

The new requirement is a debugging view of an *entire order*. Do not manually join `str(line)` values; expose a deterministic, unambiguous representation by delegating to each element’s representation.

### Worked solution — reasoning before code

Using `{self.lines!r}` delegates to the list representation, which delegates in turn to each invoice line’s `__repr__`. Because the list is copied in the constructor, later changes to the input list do not silently change the order; that defensive-copy choice is separate from overriding but matters to the example.

In [8]:
class P02Order:
    def __init__(self, lines):
        self.lines = list(lines)

    def __repr__(self):
        return f"P02Order({self.lines!r})"

    def __str__(self):
        return "; ".join(str(line) for line in self.lines) or "(empty order)"

order = P02Order([line, P02InvoiceLine("notebook", 1)])
print("friendly:", order)
print("debug:", repr(order))

friendly: 2 × A "special" pen; 1 × notebook
debug: P02Order([P02InvoiceLine('A "special" pen', 2), P02InvoiceLine('notebook', 1)])


### Verify Problem 02

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [9]:
assert str(line) == '2 × A "special" pen'
assert repr(line) in repr([line])
assert f"{line!r}" == repr(line)
assert "P02InvoiceLine" in repr(order)
assert str(P02Order([])) == "(empty order)"
print("Problem 02: all checks passed")

Problem 02: all checks passed


### Check your understanding

If `P02InvoiceLine` removed `__str__`, what would `str(line)` do? **Answer:** it would select the inherited `object.__str__`, whose fallback uses this class’s `__repr__`. A list would still use `repr(line)` as before.

## Problem 03 — Formatting flags are not an overriding hierarchy

**What you will learn:** `__str__`, `__repr__`, `__format__`, f-strings

**The problem.** A user believes that `f"{obj:short}"` automatically calls the child’s `__str__`. Inspect the actual path and implement custom formatting without recursion.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — A deliberately minimal object

Start with no override of `__format__` at all. Empty f-string format specifications are not the same experiment as a non-empty specification.

In [10]:
class P03Ticket:
    def __init__(self, number):
        self.number = number

    def __str__(self):
        return f"Ticket #{self.number}"

    def __repr__(self):
        return f"P03Ticket({self.number!r})"

ticket = P03Ticket(47)
print(f"empty spec: {ticket}")

empty spec: Ticket #47


### Predict before running

Does supplying the custom format spec `short` magically select `__str__`, fall back to `__repr__`, or fail? The error is captured to keep subsequent cells executable.

In [11]:
try:
    print(f"{ticket:short}")
except TypeError as exc:
    print(type(exc).__name__, str(exc))

TypeError unsupported format string passed to P03Ticket.__format__


### Step 2 — Understand the separate entry point

The formatting operation calls `__format__(spec)`. `object.__format__` can handle an empty specification through string conversion, but a non-empty custom spec requires a class to implement its own formatting behavior.

### Worked solution — reasoning before code

Implement `__format__` explicitly. Delegate the *empty* specification to `str(self)` and define precisely what `short` and `debug` mean. Use `str(self)` or `repr(self)` rather than formatting `self` with the same spec from inside `__format__`, which could recurse.

In [12]:
class P03FormattedTicket(P03Ticket):
    def __format__(self, spec):
        if spec == "":
            return str(self)
        if spec == "short":
            return f"T-{self.number:04d}"
        if spec == "debug":
            return repr(self)
        raise ValueError(f"Unknown format spec: {spec!r}")

formatted = P03FormattedTicket(47)
for spec in ("", "short", "debug"):
    print(f"spec={spec!r}: {format(formatted, spec)}")

spec='': Ticket #47
spec='short': T-0047
spec='debug': P03Ticket(47)


### Step 3 — Inspect conversion flags

`!s` and `!r` force conversion *before* applying the format specification. That is different from asking the original object to handle a custom spec. Do not confuse the conversion flag with a `__format__` override.

In [13]:
print(f"{formatted!s}")
print(f"{formatted!r}")
print(f"{formatted:short}")

Ticket #47
P03Ticket(47)
T-0047


### Verify Problem 03

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [14]:
assert format(formatted, "") == "Ticket #47"
assert format(formatted, "short") == "T-0047"
assert format(formatted, "debug") == "P03Ticket(47)"
try:
    format(formatted, "unsupported")
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for unsupported format spec")
print("Problem 03: all checks passed")

Problem 03: all checks passed


### Check your understanding

Does `f"{formatted!r}"` use `P03FormattedTicket.__format__`? **Answer:** conversion first creates a string from `repr(formatted)`; any subsequent formatting is applied to that resulting string, not to the original ticket.

## Problem 04 — A representation must not accidentally leak internal state

**What you will learn:** override design; exposing a public view versus a debugging view

**The problem.** A credential-bearing object is easy to print accidentally through nested collections. Implement representations that expose a safe identifier without printing the credential.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Locate the leak

The broken class stores a secret and places it in `__repr__`. This illustrates the precise exposure path, but we use a harmless dummy string.

In [15]:
class P04LeakySession:
    def __init__(self, owner, token):
        self.owner, self.token = owner, token

    def __repr__(self):
        return f"P04LeakySession({self.owner!r}, {self.token!r})"

leaky = P04LeakySession("Ada", "DEMO_TOKEN_123")
print([leaky])

[P04LeakySession('Ada', 'DEMO_TOKEN_123')]


### Predict before running

If we only added a safe `__str__`, would `print([leaky])` stop leaking the token? Think back to Problem 02 before reading on.

**Explanation.** No. The list display uses `repr(element)`, so fixing `__str__` alone leaves the leak. Any trace, collection, debug logger, or f-string with `!r` can expose the debugging representation.

### Worked solution — reasoning before code

Override both public and debugging representations deliberately. Store the token for domain behavior but never interpolate it into a representation. In this exercise the safe `repr` is descriptive, not a reconstructible Python expression; security takes priority over reconstructibility.

In [16]:
class P04SafeSession(P04LeakySession):
    def __str__(self):
        return f"Session for {self.owner}"

    def __repr__(self):
        return f"P04SafeSession(owner={self.owner!r}, token=<redacted>)"

safe = P04SafeSession("Ada", "DEMO_TOKEN_123")
print(str(safe))
print(repr(safe))
print([safe])

Session for Ada
P04SafeSession(owner='Ada', token=<redacted>)
[P04SafeSession(owner='Ada', token=<redacted>)]


### Step 2 — Think like a caller

Representations cannot protect a secret that other code chooses to print directly. The contract here is narrower and testable: neither the friendly representation nor the debugging representation (including container displays) may contain the credential.

### Verify Problem 04

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [17]:
for rendered in (str(safe), repr(safe), repr([safe]), f"{safe!r}"):
    assert "DEMO_TOKEN_123" not in rendered
assert "Ada" in str(safe)
assert "<redacted>" in repr(safe)
print("Problem 04: all checks passed")

Problem 04: all checks passed


### Check your understanding

Why not just have `__repr__ = __str__`? **Answer:** it would avoid this particular leak if `__str__` were safe, but would sacrifice a useful *distinct* debugging representation. Maintaining both explicitly makes their intended audiences obvious.

---

# Part II — Dynamic dispatch inside inherited methods

The source notebook’s important twist is that an inherited method can call an overridden method on `self`. Here we turn that observation into design and debugging exercises.

## Problem 05 — Predict a three-level nested dispatch trace

**What you will learn:** runtime `self`; inherited methods calling multiple overridden hooks

**The problem.** A base method calls a hook twice, while its subclass calls another hook. Derive the exact call order without relying on class names alone.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Start with a tiny call recorder

Every method appends its *own* label to a trace. The trace lets us distinguish the class containing the inherited method from the runtime object that supplies each hook.

In [18]:
class P05Engine:
    def __init__(self):
        self.trace = []

    def plan(self):
        self.trace.append("Engine.plan")
        first = self.step()
        second = self.finalize(first)
        return second

    def step(self):
        self.trace.append("Engine.step")
        return 3

    def finalize(self, value):
        self.trace.append("Engine.finalize")
        return value * 2

class P05Turbo(P05Engine):
    def step(self):
        self.trace.append("Turbo.step")
        return 5

class P05SafeTurbo(P05Turbo):
    def finalize(self, value):
        self.trace.append("SafeTurbo.finalize")
        return value + 1

engine = P05SafeTurbo()

### Predict before running

Predict the returned integer and the *three* trace entries. Which implementation supplies `plan`? Which object is `self` during the inherited `plan`?

In [19]:
print("result:", engine.plan())
print("trace:", engine.trace)
print("runtime type:", type(engine).__name__)

result: 6
trace: ['Engine.plan', 'Turbo.step', 'SafeTurbo.finalize']
runtime type: P05SafeTurbo


### Step 2 — Explain each lookup

`plan` comes from `P05Engine`, but is bound to a `P05SafeTurbo` instance. Therefore `self.step()` finds `P05Turbo.step`, and `self.finalize(...)` finds `P05SafeTurbo.finalize`. Neither call starts the search from the class that happened to define `plan`.

### Step 3 — Add a harder variation

What if the final subclass overrides `step` and calls `super().step()`? Predict what the trace gains and whether `plan` itself changes. The next cell supplies the solution.

### Worked solution — reasoning before code

The override may extend a parent hook without copying `plan`. The `super()` expression is inside the overriding `step`, so it continues after `P05Overclocked` in this object’s MRO and reaches `P05Turbo.step`.

In [20]:
class P05Overclocked(P05SafeTurbo):
    def step(self):
        self.trace.append("Overclocked.step")
        return super().step() + 2

overclocked = P05Overclocked()
print("result:", overclocked.plan())
print("trace:", overclocked.trace)

result: 8
trace: ['Engine.plan', 'Overclocked.step', 'Turbo.step', 'SafeTurbo.finalize']


### Verify Problem 05

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [21]:
assert engine.trace == ["Engine.plan", "Turbo.step", "SafeTurbo.finalize"]
assert overclocked.trace == ["Engine.plan", "Overclocked.step", "Turbo.step", "SafeTurbo.finalize"]
assert overclocked.plan() == 8
assert P05Overclocked.__mro__[1] is P05SafeTurbo
print("Problem 05: all checks passed")

Problem 05: all checks passed


### Check your understanding

Why is the second call to `overclocked.plan()` still 8, even though the trace grows? **Answer:** the trace records observations but the computation depends on the hook return values (7 then +1), not the existing trace length.

## Problem 06 — The dangerous override called from `__init__`

**What you will learn:** virtual-call hazards; safe initialization boundaries

**The problem.** A parent constructor calls `self.build_label()`. A child overrides that method and requires an attribute its constructor has not assigned yet. Reproduce and repair the failure.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — The apparently convenient base

Calling an overridable method from the constructor seems to avoid duplicate work. However, the object already has its final subclass type even while the base constructor is executing.

In [22]:
class P06UnsafeBase:
    def __init__(self, username):
        self.username = username
        self.label = self.build_label()

    def build_label(self):
        return self.username.upper()

class P06UnsafePremium(P06UnsafeBase):
    def __init__(self, username, tier):
        super().__init__(username)
        self.tier = tier

    def build_label(self):
        return f"{self.username.upper()} [{self.tier}]"

### Predict before running

What is the first missing attribute? The parent calls its method name, but method dispatch is based on the **actual** `P06UnsafePremium` instance.

In [23]:
try:
    P06UnsafePremium("sara", "gold")
except AttributeError as exc:
    print(type(exc).__name__, str(exc))

AttributeError 'P06UnsafePremium' object has no attribute 'tier'


### Step 2 — Explain the order precisely

In `P06UnsafePremium.__init__`, `super().__init__` runs before `self.tier = tier`. The base constructor calls `self.build_label()`, which dispatches to the child override. That override reads the not-yet-created `tier` attribute and raises `AttributeError`.

### Step 3 — Select a robust design

A parent initializer should establish parent invariants without calling overridable methods that depend on child state. One practical design is to expose a computed `label` property: callers obtain the current value *after* construction, and no cached label can become stale after a tier change.

### Worked solution — reasoning before code

Keep initialization limited to owned fields. Move dynamic behavior into a read-only property that calls a hook only when a caller requests it. The child sets `tier` through its own constructor, then can safely use the property.

In [24]:
class P06SafeBase:
    def __init__(self, username):
        self.username = username

    def build_label(self):
        return self.username.upper()

    @property
    def label(self):
        return self.build_label()

class P06SafePremium(P06SafeBase):
    def __init__(self, username, tier):
        super().__init__(username)
        self.tier = tier

    def build_label(self):
        return f"{super().build_label()} [{self.tier}]"

premium = P06SafePremium("sara", "gold")
print(premium.label)
premium.tier = "platinum"
print(premium.label)

SARA [gold]
SARA [platinum]


### Verify Problem 06

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [25]:
assert premium.label == "SARA [platinum]"
assert P06SafeBase("kai").label == "KAI"
assert "label" not in premium.__dict__
assert P06SafePremium("li", "silver").label == "LI [silver]"
print("Problem 06: all checks passed")

Problem 06: all checks passed


### Check your understanding

Could we instead assign `tier` before calling `super().__init__`? **Answer:** that may happen to repair this single example, but it couples subclass initialization to the parent’s implementation and does not solve the general hazard. Avoiding virtual calls inside initialization is the sturdier default.

## Problem 07 — The “private” override that never happens

**What you will learn:** double-underscore name mangling; intentional extension points

**The problem.** A subclass defines a method with the same written name `__stamp` as the parent, yet the inherited `render` ignores it. Diagnose why and redesign the extension point.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Write two visually identical method names

Python mangles double-leading-underscore names by their *defining class*, so these declarations do not produce the same attribute name. This is a namespace feature, not security.

In [26]:
class P07BaseLetter:
    def render(self):
        return "body | " + self.__stamp()

    def __stamp(self):
        return "BASE"

class P07SpecialLetter(P07BaseLetter):
    def __stamp(self):
        return "SPECIAL"

letter = P07SpecialLetter()

### Predict before running

Does the inherited `render()` call the subclass’s visually matching `__stamp`? Inspect both the output and the class dictionaries for evidence.

In [27]:
print(letter.render())
print("base dictionary:", [key for key in vars(P07BaseLetter) if "stamp" in key])
print("child dictionary:", [key for key in vars(P07SpecialLetter) if "stamp" in key])

body | BASE
base dictionary: ['_P07BaseLetter__stamp']
child dictionary: ['_P07SpecialLetter__stamp']


### Step 2 — Locate the real attributes

Inside the base class, `self.__stamp()` is compiled using the name `_P07BaseLetter__stamp`. The subclass declared a different name, `_P07SpecialLetter__stamp`. Therefore the intended override never replaced the attribute the base requests.

### Worked solution — reasoning before code

If subclasses are *meant* to customize behavior, use an ordinary method or a single-leading-underscore hook. The naming convention `_stamp` communicates an internal extension point but does not change normal override lookup.

In [28]:
class P07ExtensibleLetter:
    def render(self):
        return "body | " + self._stamp()

    def _stamp(self):
        return "BASE"

class P07ExtensibleSpecial(P07ExtensibleLetter):
    def _stamp(self):
        return "SPECIAL"

print(P07ExtensibleLetter().render())
print(P07ExtensibleSpecial().render())

body | BASE
body | SPECIAL


### Verify Problem 07

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [29]:
assert letter.render() == "body | BASE"
assert "_P07BaseLetter__stamp" in vars(P07BaseLetter)
assert "_P07SpecialLetter__stamp" in vars(P07SpecialLetter)
assert P07ExtensibleSpecial().render() == "body | SPECIAL"
print("Problem 07: all checks passed")

Problem 07: all checks passed


### Check your understanding

Does double-underscore naming make a method private in the access-control sense? **Answer:** no. The mangled attribute remains accessible if you know its generated name. It primarily avoids accidental naming collisions; do not use it for an overridable hook.

## Problem 08 — When an instance attribute hides a method

**What you will learn:** ordinary attribute lookup; method-name collisions; debugging call errors

**The problem.** An inherited method is perfectly valid on the class but cannot be called on one particular instance after a debugging assignment. Explain and repair the shadowing.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Observe a normal inherited call

`P08Printer.print_label` is a method. When accessed through an instance, it becomes a bound method that automatically receives that instance as `self`.

In [30]:
class P08Printer:
    def print_label(self):
        return "default label"

class P08ColorPrinter(P08Printer):
    pass

printer = P08ColorPrinter()
print(printer.print_label())

default label


### Step 2 — Create an instance-specific collision

Python’s ordinary lookup can find an instance attribute ahead of a non-data-descriptor method. The method still exists on the class; only the instance’s attribute shadows it.

In [31]:
printer.print_label = "temporarily disabled"
print("instance attribute:", printer.__dict__)
try:
    printer.print_label()
except TypeError as exc:
    print(type(exc).__name__, str(exc))

instance attribute: {'print_label': 'temporarily disabled'}
TypeError 'str' object is not callable


### Predict the diagnosis

Why does `P08Printer.print_label(printer)` still work even though `printer.print_label()` fails? The explicit class access selects the function from the class rather than the string in this instance’s dictionary.

In [32]:
print("explicit class call:", P08Printer.print_label(printer))
print("class still has method:", callable(P08Printer.print_label))

explicit class call: default label
class still has method: True


### Step 3 — Repair without hiding the symptom

Deleting the accidental instance attribute restores inherited lookup. The proper domain design is to give data and methods distinct names, such as `printer.is_disabled` rather than `printer.print_label`.

### Worked solution — reasoning before code

Remove the shadowing attribute and add state with a different name. A genuine override is defined on the subclass, rather than by writing a confusing per-instance attribute.

In [33]:
del printer.print_label

class P08RevisedColorPrinter(P08Printer):
    def __init__(self, is_disabled=False):
        self.is_disabled = is_disabled

    def print_label(self):
        if self.is_disabled:
            return "(disabled)"
        return "COLOR: " + super().print_label()

revised = P08RevisedColorPrinter()
print(printer.print_label())
print(revised.print_label())
revised.is_disabled = True
print(revised.print_label())

default label
COLOR: default label
(disabled)


### Verify Problem 08

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [34]:
assert printer.print_label() == "default label"
assert P08RevisedColorPrinter().print_label() == "COLOR: default label"
assert P08RevisedColorPrinter(True).print_label() == "(disabled)"
assert "print_label" not in printer.__dict__
print("Problem 08: all checks passed")

Problem 08: all checks passed


### Check your understanding

Would this exact collision necessarily behave the same for a `property`? **Answer:** no. Properties are data descriptors and follow different precedence rules. This problem deliberately concerns an ordinary instance method, which is a non-data descriptor.

---

# Part III — `super()`, signatures, and method-resolution subtleties

Our focus moves from finding an override to correctly extending and composing overrides. These examples use fresh scenarios and build on the previous source notebook’s inheritance chain.

## Problem 09 — Three ways to call a method are not equivalent

**What you will learn:** `self.method`, `super().method`, direct base method calls

**The problem.** A child wants to extend `describe()`. Compare a recursive self-call, a cooperative super-call, and an explicit base call; determine which preserves the chain.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Build a three-level chain

Each method decorates the text produced by the next base method. Read the methods from bottom to top and try to predict the final string.

In [35]:
class P09Document:
    def describe(self):
        return "document"

class P09SignedDocument(P09Document):
    def describe(self):
        return "signed(" + super().describe() + ")"

class P09EncryptedDocument(P09SignedDocument):
    def describe(self):
        return "encrypted(" + super().describe() + ")"

encrypted = P09EncryptedDocument()
print(encrypted.describe())

encrypted(signed(document))


### Step 2 — Reproduce a faulty explicit base call

A direct `P09Document.describe(self)` names exactly one implementation. It bypasses the intermediate `SignedDocument` override, even though the receiver remains an encrypted document.

In [36]:
class P09SkipsSignature(P09SignedDocument):
    def describe(self):
        return "encrypted(" + P09Document.describe(self) + ")"

print(P09SkipsSignature().describe())

encrypted(document)


### Step 3 — Why not `self.describe()`?

Inside `describe`, an expression `self.describe()` starts lookup at the runtime class again and can re-enter the exact same override indefinitely. We demonstrate safely by limiting our own recursion and catching the resulting exception.

In [37]:
class P09Recursive(P09SignedDocument):
    def describe(self):
        return "encrypted(" + self.describe() + ")"

try:
    P09Recursive().describe()
except RecursionError:
    print("RecursionError: self.describe() re-enters the override")

RecursionError: self.describe() re-enters the override


### Worked solution — reasoning before code

Use `super().describe()` in an extending override. This continues from the owner of that `super()` expression through the receiver’s current MRO, preserving the intervening implementations that also cooperate.

In [38]:
print("MRO:", [cls.__name__ for cls in P09EncryptedDocument.__mro__])
print("cooperative:", encrypted.describe())
print("hardcoded:", P09SkipsSignature().describe())

MRO: ['P09EncryptedDocument', 'P09SignedDocument', 'P09Document', 'object']
cooperative: encrypted(signed(document))
hardcoded: encrypted(document)


### Verify Problem 09

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [39]:
assert encrypted.describe() == "encrypted(signed(document))"
assert P09SkipsSignature().describe() == "encrypted(document)"
assert P09SignedDocument().describe() == "signed(document)"
print("Problem 09: all checks passed")

Problem 09: all checks passed


### Check your understanding

Is a direct parent call *always* wrong? **Answer:** no: it may be intentional when you specifically want that implementation. It is risky for extensible or multiple-inheritance designs, where preserving the cooperative chain is the goal.

## Problem 10 — `super()` is anchored in the method, not the call site

**What you will learn:** lexical `super` anchor; reuse inherited methods; runtime MRO

**The problem.** Two descendants inherit the same method containing `super()`. Show that its next destination is determined by the class where the method was defined *and* the actual object’s MRO.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Construct two chains with shared code

`P10Middle.greet` has `super().greet()` written inside the `Middle` class. Subclasses can inherit this implementation unchanged.

In [40]:
class P10Root:
    def greet(self):
        return "root"

class P10Middle(P10Root):
    def greet(self):
        return "middle -> " + super().greet()

class P10Child(P10Middle):
    pass

print(P10Child().greet())

middle -> root


### Step 2 — Add an extra class *after* Middle in a valid MRO

Use a separate mixin and a diamond: the final class has `Middle` followed by `Addon` followed by `Root`. Crucially, the inherited source code of `Middle.greet` is unchanged.

In [41]:
class P10Addon(P10Root):
    def greet(self):
        return "addon -> " + super().greet()

class P10Combined(P10Middle, P10Addon):
    pass

print([cls.__name__ for cls in P10Combined.__mro__])
print(P10Combined().greet())

['P10Combined', 'P10Middle', 'P10Addon', 'P10Root', 'object']
middle -> addon -> root


### Step 3 — Explain the changed destination

When called on `P10Child`, the next class after `P10Middle` is `P10Root`. When called on `P10Combined`, the next class after `P10Middle` is `P10Addon`. Therefore treating `super()` as a hardcoded alias for `P10Root` would give the wrong prediction.

### Step 4 — Confirm by a direct call

Call `P10Middle.greet` on a combined instance. Although you chose the function explicitly, `self` is still a combined instance, and the `super()` within that function follows its MRO.

In [42]:
combined = P10Combined()
print(P10Middle.greet(combined))
print("same function as inherited?", P10Combined.greet is P10Middle.greet)

middle -> addon -> root
same function as inherited? True


### Worked solution — reasoning before code

The correct tracing procedure is: (1) identify the function containing `super()`, (2) identify the runtime object and its MRO, and (3) continue *after the function’s owner* in that MRO. In ordinary instance methods, zero-argument `super()` handles these pieces automatically.

### Verify Problem 10

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [43]:
assert P10Child().greet() == "middle -> root"
assert combined.greet() == "middle -> addon -> root"
assert P10Middle.greet(combined) == combined.greet()
assert P10Combined.__mro__.index(P10Addon) < P10Combined.__mro__.index(P10Root)
print("Problem 10: all checks passed")

Problem 10: all checks passed


### Check your understanding

If we changed `P10Middle.greet` to call `P10Root.greet(self)` directly, which text would disappear for a combined instance? **Answer:** `addon ->` would disappear; the direct class call would bypass `P10Addon`.

## Problem 11 — A subclass can break an inherited caller without overriding it

**What you will learn:** signature compatibility; keyword arguments; substitutability

**The problem.** A parent’s `publish()` calls `self.transform(text, *, uppercase=...)`. A child overrides `transform` but omits the keyword parameter. Locate the precise failure and preserve the contract.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Establish the base client contract

The inherited `publish` method knows only that `transform` accepts a string and an `uppercase` keyword. That expectation applies equally to subclasses.

In [44]:
class P11Publisher:
    def transform(self, text, *, uppercase=False):
        return text.upper() if uppercase else text

    def publish(self, text):
        return self.transform(text, uppercase=True)

print(P11Publisher().publish("hello"))

HELLO


### Step 2 — Add an incompatible override

The child author focuses only on direct calls such as `transform("hello")`. The inherited caller supplies an optional keyword that the child no longer accepts.

In [45]:
class P11BrokenPublisher(P11Publisher):
    def transform(self, text):
        return "[" + text + "]"

try:
    print(P11BrokenPublisher().publish("hello"))
except TypeError as exc:
    print(type(exc).__name__, str(exc))

TypeError P11BrokenPublisher.transform() got an unexpected keyword argument 'uppercase'


### Step 3 — Reason about behavior, not just names

Matching the method name is insufficient. Inherited and external clients rely on an accepted calling convention. A child override should support the base method’s promised inputs unless the interface is explicitly redesigned.

### Worked solution — reasoning before code

Keep the keyword and extend the result of the parent transformation via `super()`. The child adds decoration *after* the uppercase conversion. This keeps both the old call forms functional and avoids duplicating base logic.

In [46]:
class P11DecoratedPublisher(P11Publisher):
    def transform(self, text, *, uppercase=False):
        transformed = super().transform(text, uppercase=uppercase)
        return "[" + transformed + "]"

publisher = P11DecoratedPublisher()
print(publisher.transform("hello"))
print(publisher.transform("hello", uppercase=True))
print(publisher.publish("hello"))

[hello]
[HELLO]
[HELLO]


### Verify Problem 11

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [47]:
assert publisher.transform("hello") == "[hello]"
assert publisher.transform("hello", uppercase=True) == "[HELLO]"
assert publisher.publish("hello") == "[HELLO]"
assert P11Publisher().publish("hello") == "HELLO"
print("Problem 11: all checks passed")

Problem 11: all checks passed


### Check your understanding

Would adding `**kwargs` while ignoring `uppercase` fully solve the interface problem? **Answer:** it might prevent this `TypeError` but could silently violate the expected uppercase behavior. Compatibility includes semantics, not just parameter acceptance.

## Problem 12 — The overridden method and the captured bound method

**What you will learn:** bound methods; dynamic lookup; controlled class patching

**The problem.** A callback is saved before a subclass’s method is changed. Does the saved callback automatically update? Contrast stored bound methods with fresh attribute lookup.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Capture a bound method

`callback = obj.status` resolves and binds a concrete function at capture time. We compare this saved callable with a fresh `obj.status` lookup later. This is an isolated educational experiment, not a recommendation to patch production classes.

In [48]:
class P12Sensor:
    def status(self):
        return "normal"

sensor = P12Sensor()
saved_callback = sensor.status
print(saved_callback())
print(sensor.status())
print(saved_callback.__self__ is sensor)

normal
normal
True


### Predict before running

We will replace the class’s `status` attribute with a new function. Write down whether `saved_callback()` and `sensor.status()` should now agree.

In [49]:
old_status = P12Sensor.status

def p12_replacement(self):
    return "updated"

P12Sensor.status = p12_replacement
try:
    print("saved:", saved_callback())
    print("fresh:", sensor.status())
finally:
    P12Sensor.status = old_status

saved: normal
fresh: updated


### Step 2 — Explain the two results

The saved bound method retained the earlier function and receiver. A fresh attribute lookup on `sensor` saw the temporarily replaced class attribute. Restoring the class in `finally` avoids contaminating later exercises.

### Step 3 — Connect this back to overriding

When designing an extensible object, decide whether a saved callback should preserve the implementation chosen at registration time or route to the currently visible override at invocation time. A wrapper using `self.status()` provides the latter behavior.

### Worked solution — reasoning before code

Store a small wrapper that looks up `self.status` at call time. This is often useful for callback registries that should respect later replacement of a method; for a fixed callback, storing the bound method is equally valid.

In [50]:
dynamic_callback = lambda: sensor.status()
print("after restoration:", dynamic_callback())
P12Sensor.status = p12_replacement
try:
    print("saved old function:", saved_callback())
    print("dynamic wrapper:", dynamic_callback())
finally:
    P12Sensor.status = old_status

after restoration: normal
saved old function: normal
dynamic wrapper: updated


### Verify Problem 12

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [51]:
assert saved_callback() == "normal"
assert dynamic_callback() == "normal"
assert sensor.status() == "normal"
assert saved_callback.__self__ is sensor
print("Problem 12: all checks passed")

Problem 12: all checks passed


### Check your understanding

If the method belongs to a *subclass* and you saved a bound instance callback, is this still a distinction? **Answer:** yes. The example demonstrates function capture versus later attribute lookup; inherited versus locally declared methods does not eliminate that distinction.

---

# Part IV — Harder API boundaries and cooperative composition

The following problems examine how overriding interacts with special lookup, class APIs, and multiple inheritance. These are extensions beyond the supplied lecture, using its central rules as the foundation.

## Problem 13 — A method override cannot replace an interface with a property invisibly

**What you will learn:** method/property mismatch; caller-facing contracts

**The problem.** A subclass replaces a callable `summary()` method with a `@property` named `summary`. Existing clients still call `obj.summary()`. Diagnose the failure and implement an interface-preserving design.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Identify what users of the class know

The caller has been promised a callable method, not merely “something named summary.” The base class passes its own tests.

In [52]:
class P13Record:
    def __init__(self, label):
        self.label = label

    def summary(self):
        return f"Record: {self.label}"

def p13_client(record):
    return record.summary()

print(p13_client(P13Record("alpha")))

Record: alpha


### Step 2 — Introduce a seemingly harmless API change

The subclass’s property returns a string when accessed, but the inherited client attempts to *call* that string. This is a behavioral break even though the attribute name is identical.

In [53]:
class P13BrokenRecord(P13Record):
    @property
    def summary(self):
        return f"Special: {self.label}"

broken = P13BrokenRecord("alpha")
print("attribute access:", broken.summary)
try:
    p13_client(broken)
except TypeError as exc:
    print(type(exc).__name__, str(exc))

attribute access: Special: alpha
TypeError 'str' object is not callable


### Step 3 — Choose a compatible extension point

If the public interface is `summary()`, keep it callable. A separate property may provide convenient read-only access with a *different* name. Use a simple override that preserves expected invocation syntax.

### Worked solution — reasoning before code

The corrected subclass overrides `summary` with an ordinary method. The optional `summary_text` property forwards to that method, so it also picks up a further subclass override through `self.summary()`.

In [54]:
class P13SpecialRecord(P13Record):
    def summary(self):
        return f"Special: {self.label}"

    @property
    def summary_text(self):
        return self.summary()

class P13UpperRecord(P13SpecialRecord):
    def summary(self):
        return super().summary().upper()

special = P13SpecialRecord("alpha")
upper = P13UpperRecord("alpha")
print(p13_client(special))
print(upper.summary_text)

Special: alpha
SPECIAL: ALPHA


### Verify Problem 13

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [55]:
assert p13_client(special) == "Special: alpha"
assert upper.summary_text == "SPECIAL: ALPHA"
assert callable(special.summary)
assert not callable(broken.summary)
print("Problem 13: all checks passed")

Problem 13: all checks passed


### Check your understanding

Why does `summary_text` on `P13UpperRecord` show upper-case text although the property was defined in its parent? **Answer:** the inherited property’s getter calls `self.summary()`, and `self` is an upper-record instance; normal dispatch selects its override.

## Problem 14 — Class-level constructors and polymorphic inherited helpers

**What you will learn:** classmethod binding; factories; preserving subclass identity

**The problem.** An inherited factory should produce instances of the concrete class through which it is called. Then a child overrides a hook used by the factory without duplicating factory logic.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Create an inherited classmethod factory

The factory uses `cls`, not a hardcoded base class, and delegates configuration to a classmethod hook. This makes both the constructor target and the chosen configuration polymorphic.

In [56]:
class P14Connection:
    def __init__(self, host, port):
        self.host, self.port = host, port

    @classmethod
    def default_port(cls):
        return 80

    @classmethod
    def from_host(cls, host):
        return cls(host, cls.default_port())

class P14SecureConnection(P14Connection):
    @classmethod
    def default_port(cls):
        return 443

plain = P14Connection.from_host("example.test")
secure = P14SecureConnection.from_host("example.test")
print(type(plain).__name__, plain.port)
print(type(secure).__name__, secure.port)

P14Connection 80
P14SecureConnection 443


### Predict the next twist

`from_host` is *not* overridden in `P14SecureConnection`; why does it still use port 443? The `cls` parameter is the concrete class on which the classmethod was invoked.

### Step 2 — Add a classmethod override that extends the base

Now define a subclass with a custom constructor call but preserve the base’s shared parsing and default port behavior. Avoid assuming that `super().from_host(...)` forces construction of the base class.

### Worked solution — reasoning before code

Inside the subclass classmethod, `super().from_host(host)` calls the next classmethod but continues binding `cls` to the invoking subclass. The returned object therefore retains the most-derived type. Set subclass-specific state only after the shared factory has created the instance.

In [57]:
class P14TaggedSecureConnection(P14SecureConnection):
    @classmethod
    def from_host(cls, host):
        result = super().from_host(host.strip())
        result.tag = "tls"
        return result

tagged = P14TaggedSecureConnection.from_host("  api.test  ")
print(type(tagged).__name__, tagged.host, tagged.port, tagged.tag)
print([c.__name__ for c in type(tagged).__mro__])

P14TaggedSecureConnection api.test 443 tls
['P14TaggedSecureConnection', 'P14SecureConnection', 'P14Connection', 'object']


### Step 3 — Identify a hidden precondition

Subclass-specific post-construction mutation is safe here because the base factory returns the instance before the tag is assigned. If downstream users require `tag` during `__init__`, an explicit, stable constructor contract would be preferable. Factories should document such invariants.

### Verify Problem 14

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [58]:
assert type(plain) is P14Connection
assert type(secure) is P14SecureConnection
assert secure.port == 443
assert type(tagged) is P14TaggedSecureConnection
assert (tagged.host, tagged.port, tagged.tag) == ("api.test", 443, "tls")
print("Problem 14: all checks passed")

Problem 14: all checks passed


### Check your understanding

What changes if `from_host` returns `P14Connection(host, ...)` instead of `cls(host, ...)`? **Answer:** calling the factory on a subclass still returns a base instance, defeating subtype-preserving construction.

## Problem 15 — Trace cooperative wrappers in two different orders

**What you will learn:** multiple inheritance; C3 MRO; value transformation

**The problem.** Two mixins wrap a method with different transformations. Investigate how base order changes the result, then formulate a precise method-tracing rule.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Set up one concrete method and two mixins

Each mixin calls `super().read()` exactly once and decorates the returned value. They do not assume which class comes next; the composed class defines the order.

In [59]:
class P15DataSource:
    def read(self):
        return "data"

class P15BracketMixin:
    def read(self):
        return "[" + super().read() + "]"

class P15UpperMixin:
    def read(self):
        return super().read().upper()

class P15BracketThenUpper(P15BracketMixin, P15UpperMixin, P15DataSource):
    pass

class P15UpperThenBracket(P15UpperMixin, P15BracketMixin, P15DataSource):
    pass

### Predict before running

Both classes have the same three ancestors but list them in a different order. Predict their MROs and outputs. Here the transformations happen to commute for these characters; we will make the distinction visible next.

In [60]:
for cls in (P15BracketThenUpper, P15UpperThenBracket):
    print(cls.__name__, [c.__name__ for c in cls.__mro__])
    print("result:", cls().read())

P15BracketThenUpper ['P15BracketThenUpper', 'P15BracketMixin', 'P15UpperMixin', 'P15DataSource', 'object']
result: [DATA]
P15UpperThenBracket ['P15UpperThenBracket', 'P15UpperMixin', 'P15BracketMixin', 'P15DataSource', 'object']
result: [DATA]


### Step 2 — Reveal an order-sensitive composition

Uppercasing bracketed text does not change brackets, so both results look the same. Replace a wrapper with a suffix that depends on the input’s current case. This is a useful warning: equal outputs alone do not prove equal dispatch paths.

In [61]:
class P15CaseTagMixin:
    def read(self):
        text = super().read()
        return text + ("-UP" if text.isupper() else "-low")

class P15TagOutside(P15CaseTagMixin, P15UpperMixin, P15DataSource):
    pass

class P15UpperOutside(P15UpperMixin, P15CaseTagMixin, P15DataSource):
    pass

print("tag outside:", P15TagOutside().read())
print("uppercase outside:", P15UpperOutside().read())

tag outside: DATA-UP
uppercase outside: DATA-LOW


### Step 3 — Trace call and return phases separately

For `P15TagOutside`, tag sees uppercase `DATA`, so returns `DATA-UP`. For `P15UpperOutside`, tag first sees lowercase `data`, returns `data-low`, and the outer uppercase wrapper changes that whole result to `DATA-LOW`. The return phase unwinds in the reverse order of entering the wrappers.

### Worked solution — reasoning before code

Cooperative mixins should (1) use `super()`, (2) call the next method a deliberate number of times, normally once, and (3) have an explicit shared signature and return type. Test both the MRO and the observable effect when order is semantically important.

### Verify Problem 15

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [62]:
assert P15BracketThenUpper().read() == "[DATA]"
assert P15UpperThenBracket().read() == "[DATA]"
assert P15TagOutside().read() == "DATA-UP"
assert P15UpperOutside().read() == "DATA-LOW"
assert P15TagOutside.__mro__.index(P15CaseTagMixin) < P15TagOutside.__mro__.index(P15UpperMixin)
print("Problem 15: all checks passed")

Problem 15: all checks passed


### Check your understanding

What does it mean that “`super()` is not simply my parent”? **Answer:** `super()` proceeds to the next applicable class in the current object’s MRO. In a composed class, that next class may be a sibling mixin, not the base named in the mixin’s own definition.

---

# Part V — Integrated capstone: make overriding maintainable

The final exercise integrates overriding, initialization safety, separate human/debug representations, runtime hooks, validation, and an inherited algorithm. Its scenario and requirements are new, and it is deliberately developed one layer at a time.

## Problem 16 — Capstone: a reusable task executor with overridable behavior

**What you will learn:** template method; validation; runtime hooks; `super()`; observability; representations

**The problem.** Build a small in-memory executor that processes a nonempty task name, allows a child to customize execution, allows another child to wrap success, and never records a success event for a rejected input.

**Notebook workflow:** Examine the starting implementation, predict the next result **before** executing the next cell, observe the result, explain the lookup path, and then implement and test a fix. The solution is intentionally revealed in stages rather than all at once.

### Step 1 — Specify the contract in words

The common `run(task)` method should trim input, reject an empty name with `ValueError`, call `self.execute(clean_task)`, append the successful result to `history`, and return that result. The base constructor must initialize its own state without calling any overridable hooks. Each subclass may extend `execute` but must not duplicate `run`.

**Before writing code, derive two invariants:**

1. A failed `run("   ")` leaves `history` unchanged.
2. When `run` is inherited by a subclass, its internal `self.execute(...)` must still dispatch to the subclass’s override.

We will separately test what happens if the override itself raises. In this design history records **successful** executions only, so an exception from `execute` must not append anything.

### Step 2 — Build the parent algorithm

There is one validation point and one place where a successful result is recorded. Writing `execute` as a normal hook gives subclasses an explicit extension point. Notice that the constructor sets `history` directly rather than invoking that hook.

In [63]:
class P16Executor:
    def __init__(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("executor name must be nonempty")
        self.name = name.strip()
        self.history = []

    def execute(self, task):
        return f"done:{task}"

    def run(self, task):
        if not isinstance(task, str):
            raise TypeError("task must be a string")
        clean_task = task.strip()
        if not clean_task:
            raise ValueError("task must be nonempty")
        result = self.execute(clean_task)
        self.history.append(result)
        return result

    def __str__(self):
        return f"{self.name}: {len(self.history)} completed"

    def __repr__(self):
        return f"{type(self).__name__}(name={self.name!r}, completed={len(self.history)})"

base_executor = P16Executor(" primary ")
print(base_executor.run("  compile  "))
print(str(base_executor))
print(repr(base_executor))

done:compile
primary: 1 completed
P16Executor(name='primary', completed=1)


### Step 3 — Predict polymorphic behavior

Implementing `run` only once should be enough: a subclass overriding `execute` changes what the inherited `run` does. Trace the runtime receiver before moving on to the next cell.

In [64]:
class P16UpperExecutor(P16Executor):
    def execute(self, task):
        return super().execute(task.upper())

upper_executor = P16UpperExecutor("uppercase")
print(upper_executor.run(" check data "))
print(upper_executor.history)
print(upper_executor)

done:CHECK DATA
['done:CHECK DATA']
uppercase: 1 completed


### Step 4 — Add a subclass that wraps successful execution

A timed or external worker is unnecessary for this example; instead, we add a deterministic audit trail. The subclass calls `super().execute` and then records an audit event. This is an `execute`-level event, not the authoritative `history` record written by `run`.

In [65]:
class P16AuditedExecutor(P16UpperExecutor):
    def __init__(self, name):
        super().__init__(name)
        self.audit = []

    def execute(self, task):
        result = super().execute(task)
        self.audit.append((task, result))
        return result

    def __repr__(self):
        return (f"{type(self).__name__}(name={self.name!r}, "
                f"completed={len(self.history)}, audits={len(self.audit)})")

audited = P16AuditedExecutor("review")
print(audited.run("  archive  "))
print(audited.audit)
print(repr(audited))

done:ARCHIVE
[('archive', 'done:ARCHIVE')]
P16AuditedExecutor(name='review', completed=1, audits=1)


### Step 5 — Debug the edge cases without interrupting execution

A sound implementation is defined by how it handles failure as well as success. We exercise invalid input and show that validation occurs before execution; no audit entry or completion history should be added.

In [66]:
before = (len(audited.history), len(audited.audit))
for invalid_task in ("   ", None, 123):
    try:
        audited.run(invalid_task)
    except (ValueError, TypeError) as exc:
        print(f"rejected {invalid_task!r}: {type(exc).__name__}")
print("counts before:", before)
print("counts after:", (len(audited.history), len(audited.audit)))

rejected '   ': ValueError
rejected None: TypeError
rejected 123: TypeError
counts before: (1, 1)
counts after: (1, 1)


### Step 6 — What if the overridden hook fails?

The validation may succeed but the domain hook can still raise. Because `history.append` occurs *after* the hook returns, the parent records no false success. In this design, the audit subclass also appends only after its parent hook returns.

In [67]:
class P16FailingExecutor(P16AuditedExecutor):
    def execute(self, task):
        if task == "FAIL":
            raise RuntimeError("simulated failure")
        return super().execute(task)

failing = P16FailingExecutor("resilient")
try:
    failing.run("FAIL")
except RuntimeError as exc:
    print(type(exc).__name__, str(exc))
print("history:", failing.history, "audit:", failing.audit)
print("recovery:", failing.run("ok"))

RuntimeError simulated failure
history: [] audit: []
recovery: done:OK


### Worked solution — reasoning before code

The template method pattern puts stable workflow (`run`) in the parent and variable behavior (`execute`) in overridable hooks. The subclass uses `super()` to extend the immediate behavior *as resolved by MRO*, not to copy the algorithm. Human-friendly `__str__` and informative `__repr__` have separate responsibilities. No initializer calls a virtual hook before all subclass fields exist.

### Step 7 — Verify the full behavioral contract

These tests cover base behavior, dispatched behavior, successful audit, invalid input, hook failure, recovery, and useful string representations. They are more valuable than checking only the final printed success message.

### Verify Problem 16

These assertions encode the behavioral contract. Run the cell: if an assertion fails, compare the expected result with the method lookup or initialization trace. A green run prints the confirmation line.

In [68]:
assert base_executor.history == ["done:compile"]
assert upper_executor.history == ["done:CHECK DATA"]
assert audited.history == ["done:ARCHIVE"]
assert audited.audit == [("archive", "done:ARCHIVE")]
assert before == (len(audited.history), len(audited.audit))
assert failing.history == ["done:OK"]
assert failing.audit == [("ok", "done:OK")]
assert "1 completed" in str(audited)
assert "P16AuditedExecutor" in repr(audited)
assert P16Executor("empty").history == []
print("Problem 16: all checks passed")

Problem 16: all checks passed


### Check your understanding

Why can `P16AuditedExecutor` inherit `run` unchanged and still customize execution? **Answer:** `run` calls `self.execute`, so the lookup begins with the actual audited instance, not with `P16Executor`. Why append to history *after* the hook call? **Answer:** it prevents a thrown exception from creating a success record.

---

# Final tutorial review: explain the mechanism, not only the output

Before opening the answers below, try to answer each question in one or two sentences. Use the experiments above to verify, rather than relying on memory.

| Question | Answer |
|---|---|
| Why is returning a float from `__str__` not the same as returning its printable value? | The protocol requires a `str`; Python does not automatically coerce an invalid return value. |
| Why can a list expose information omitted from `str(element)`? | A list builds its representation from each element’s `repr`. |
| Can an inherited method call a child's override? | Yes, via `self.hook()`; `self` is the runtime object. |
| What makes calling a hook in a base constructor dangerous? | The child override can run before the child has initialized its fields. |
| Why do two `__hidden` method definitions fail to override one another? | Each is name-mangled with a different declaring class name. |
| What does `super()` choose in a diamond? | The next applicable implementation after the defining class in the receiver’s MRO. |
| Can the same method name still break a public interface? | Yes: incompatible signatures or replacing a method with a property breaks callers. |
| Why can two mixin orders yield different values? | Call order and return-time wrapper transformations follow their respective MROs. |

**Practice task (without a provided code solution):** Add a `P16DryRunExecutor` that overrides `execute` to return `"would:" + task` without recording a different kind of event. Prove that it still inherits base validation and success-only history, and explain why no override of `run` is necessary. This is intentionally left as a final self-test after 16 fully solved problems.

In [69]:
# Optional self-test starter. Uncomment after implementing your own P16DryRunExecutor.
# dry = P16DryRunExecutor("preview")
# assert dry.run("  deploy  ") == "would:deploy"
# assert dry.history == ["would:deploy"]
# print("Optional self-test passed")